In [1]:
#Libraries
import os, sys
import comet_ml
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as nnf
from torch.utils.data import DataLoader, TensorDataset, random_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, classification_report
import logging
logging.getLogger("comet_ml").setLevel(logging.CRITICAL)
import warnings
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.preprocessing import StandardScaler
from tsaug import TimeWarp
from collections import defaultdict

#Hide warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)


In [2]:
PROJECT_ROOT = 'C:/Users/sebas/Desktop'
sys.path.insert(0,PROJECT_ROOT)

#Params to use
# from configs.xxx_params import params

from configs.cnn_params import params




# Reading data
file_path = r'C:/Users/sebas/Desktop/data/train.pickle' # Write the path of train.pickle

# Reading data
df = pd.read_pickle(file_path)

test_path = r'C:/Users/sebas/Desktop/data/test.pickle' # Write the path of train.pickle
df_test = pd.read_pickle(test_path)

In [3]:
# Please provide your comet API key under your username. This is utilized to track experiments on comet.

#API_dict = {"User1": "API_user1",
#           "User2": "API_user2"}

API_dict = {"Nikita": "7WU7vFXt4DN7SLBWH9YJJOBta"}

# **Functions**

## **1.1 Models**

The following cell cointains the models throughout the project and different tasks.

It includes normal FCN, CNN, and LSTM for **Task 1**.
It also includes the experimental models BinaryHeadLSTM, BinaryHeadFCN, BinaryHeadCNN, PhysicalFeatureHeadLSTM, PhysicalFeatureHeadFCN, and PhysicalFeatureHeadCNN. This models were used for **Additional task 2**.

In [4]:
class LSTMClassifier(nn.Module):
  def __init__(self, input_dim, hidden_dim=64, num_layers=1, num_classes = 12):
    super(LSTMClassifier, self).__init__()
    self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
    self.fc = nn.Linear(hidden_dim, num_classes)

  def forward(self, x):
    _, (hn, _) = self.lstm(x)
    out = self.fc(hn[-1])     # last hidden state
    return out

class CNNClassifier(nn.Module):
  def __init__(self, X_shape, conv1_out = 16, dropout = 0.3, conv2_out = 32, fc_hidden = 32):
    super(CNNClassifier, self).__init__()
    self.conv1 = nn.Conv2d(1, conv1_out, kernel_size=(1,6), padding=(1,0))
    self.bn1 = nn.BatchNorm2d(conv1_out)
    self.pool1 = nn.MaxPool2d((2,1))
    self.drop1 = nn.Dropout(dropout)

    self.conv2 = nn.Conv2d(conv1_out, conv2_out, kernel_size=(3,1), padding=(1,0))
    self.bn2 = nn.BatchNorm2d(conv2_out)
    self.pool2 = nn.MaxPool2d((2,1))
    self.drop2 = nn.Dropout(dropout)

    h_out = X_shape[2] // 4
    w_out = 1
    self.flatten = nn.Flatten()
    self.fc1 = nn.Linear(conv2_out * h_out * w_out, fc_hidden)
    self.drop3 = nn.Dropout(dropout)
    self.fc2 = nn.Linear(fc_hidden, 12)  # Number of classes

  def forward(self, x):
    x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
    x = self.drop1(x)
    x = self.pool2(torch.relu(self.bn2(self.conv2(x))))
    x = self.drop2(x)
    x = self.flatten(x)
    x = torch.relu(self.fc1(x))
    x = self.drop3(x)
    x = self.fc2(x)
    return x

class FCNClassifier(nn.Module):
  def __init__(self, in_channels = 6, conv1_out = 64, conv2_out = 128, conv3_out = 128, num_classes = 12):
    super(FCNClassifier, self).__init__()
    self.conv1 = nn.Conv1d(in_channels, conv1_out, kernel_size=8, padding=4)
    self.bn1 = nn.BatchNorm1d(conv1_out)
    self.conv2 = nn.Conv1d(conv1_out, conv2_out, kernel_size=5, padding=2)
    self.bn2 = nn.BatchNorm1d(conv2_out)
    self.conv3 = nn.Conv1d(conv2_out, conv3_out, kernel_size=3, padding=1)
    self.bn3 = nn.BatchNorm1d(conv3_out)
    self.global_pool = nn.AdaptiveAvgPool1d(1)
    self.fc = nn.Linear(conv3_out, num_classes)

  def forward(self, x):
    x = x.permute(0, 2, 1)
    x = torch.relu(self.bn1(self.conv1(x)))
    x = torch.relu(self.bn2(self.conv2(x)))
    x = torch.relu(self.bn3(self.conv3(x)))
    x = self.global_pool(x).squeeze(-1)
    x = self.fc(x)
    return x

#Models with binary classifaction head
class BinaryHeadLSTM(nn.Module):
  def __init__(self, input_dim, hidden_dim=64, num_layers=1, num_classes = 12):
    super(BinaryHeadLSTM, self).__init__()
    self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
    self.binary_fc = nn.Linear(hidden_dim, 2)
    self.multi_fc = nn.Linear(hidden_dim, num_classes)

  def forward(self, x):
    _, (hn, _) = self.lstm(x)
    z = hn[-1]

    logits_b = self.binary_fc(z)
    logits_m = self.multi_fc(z)

    return logits_b, logits_m

class BinaryHeadCNN(nn.Module):
  def __init__(self, X_shape, conv1_out = 16, dropout = 0.3, conv2_out = 32, fc_hidden = 32):
    super(BinaryHeadCNN, self).__init__()
    self.conv1 = nn.Conv2d(1, conv1_out, kernel_size=(1,6), padding=(1,0))
    self.bn1 = nn.BatchNorm2d(conv1_out)
    self.pool1 = nn.MaxPool2d((2,1))
    self.drop1 = nn.Dropout(dropout)

    self.conv2 = nn.Conv2d(conv1_out, conv2_out, kernel_size=(3,1), padding=(1,0))
    self.bn2 = nn.BatchNorm2d(conv2_out)
    self.pool2 = nn.MaxPool2d((2,1))
    self.drop2 = nn.Dropout(dropout)

    h_out = X_shape[2] // 4
    w_out = 1
    self.flatten = nn.Flatten()
    self.shared_fc = nn.Linear(conv2_out * h_out * w_out, fc_hidden)
    self.drop3 = nn.Dropout(dropout)

    self.binary_fc = nn.Linear(fc_hidden, 2)
    self.multi_fc = nn.Linear(fc_hidden, 12)

  def forward(self, x):
    x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
    x = self.drop1(x)
    x = self.pool2(torch.relu(self.bn2(self.conv2(x))))
    x = self.drop2(x)
    x = self.flatten(x)
    x = torch.relu(self.shared_fc(x))
    x = self.drop3(x)
    logits_b = self.binary_fc(x)
    logits_m = self.multi_fc(x)
    return logits_b, logits_m

class BinaryHeadFCN(nn.Module):
  def __init__(self, in_channels = 6, conv1_out = 64, conv2_out = 128, conv3_out = 128, fc_hidden = 128, dropout = 0.3, num_classes = 12):
    super(BinaryHeadFCN, self).__init__()
    #1D conv backbone
    self.conv1 = nn.Conv1d(in_channels, conv1_out, kernel_size=8, padding=4)
    self.bn1   = nn.BatchNorm1d(conv1_out)

    self.conv2 = nn.Conv1d(conv1_out, conv2_out, kernel_size=5, padding=2)
    self.bn2   = nn.BatchNorm1d(conv2_out)

    self.conv3 = nn.Conv1d(conv2_out, conv3_out, kernel_size=3, padding=1)
    self.bn3   = nn.BatchNorm1d(conv3_out)

    #global pooling collapses time dimension
    self.global_pool = nn.AdaptiveAvgPool1d(1)

    #shared dense layer
    self.shared_fc = nn.Linear(conv3_out, fc_hidden)
    self.drop      = nn.Dropout(dropout)

    #two heads
    self.binary_fc = nn.Linear(fc_hidden, 2)
    self.multi_fc  = nn.Linear(fc_hidden, num_classes)

  def forward(self, x):
    # x: (batch, time, features) → (batch, features, time)
    x = x.permute(0, 2, 1)

    # conv stack
    x = torch.relu(self.bn1(self.conv1(x)))
    x = torch.relu(self.bn2(self.conv2(x)))
    x = torch.relu(self.bn3(self.conv3(x)))

    # global pooling → (batch, conv3_out)
    x = self.global_pool(x).squeeze(-1)

    # shared FC + dropout
    h = torch.relu(self.shared_fc(x))
    h = self.drop(h)

    # binary head and multi-class head
    logits_b = self.binary_fc(h)
    logits_m = self.multi_fc(h)

    return logits_b, logits_m

#Models with physical features prediction head
class PhysicalFeatureHeadLSTM(nn.Module):
  def __init__(self, input_dim, hidden_dim=64, num_layers=1, num_classes = 12):
    super(PhysicalFeatureHeadLSTM, self).__init__()
    self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)

    self.classifier = nn.Linear(hidden_dim, num_classes)

    # Classification head
    self.classifier = nn.Linear(hidden_dim, num_classes)
    # Phys regression head
    self.regressor  = nn.Linear(hidden_dim, 3)

  def forward(self, x):
    # x shape: (batch, seq_len, features)
    _, (hn, _) = self.lstm(x)
    h_last = hn[-1]

    class_logits = self.classifier(h_last)
    phys_preds = self.regressor(h_last)

    return class_logits, phys_preds

class PhysicalFeatureHeadCNN(nn.Module):
  def __init__(self, X_shape, conv1_out = 16, dropout = 0.3, conv2_out = 32, fc_hidden = 32):
    super(PhysicalFeatureHeadCNN, self).__init__()
    self.conv1 = nn.Conv2d(1, conv1_out, kernel_size=(1,6), padding=(1,0))
    self.bn1 = nn.BatchNorm2d(conv1_out)
    self.pool1 = nn.MaxPool2d((2,1))
    self.drop1 = nn.Dropout(dropout)

    self.conv2 = nn.Conv2d(conv1_out, conv2_out, kernel_size=(3,1), padding=(1,0))
    self.bn2 = nn.BatchNorm2d(conv2_out)
    self.pool2 = nn.MaxPool2d((2,1))
    self.drop2 = nn.Dropout(dropout)

    h_out = X_shape[2] // 4
    w_out = 1
    self.flatten = nn.Flatten()
    self.fc_shared = nn.Linear(conv2_out * h_out * w_out, fc_hidden)
    self.drop3 = nn.Dropout(dropout)

    self.fc_class = nn.Linear(fc_hidden, 12)
    self.fc_phys = nn.Linear(fc_hidden, 3)

  def forward(self, x):
    x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
    x = self.drop1(x)
    x = self.pool2(torch.relu(self.bn2(self.conv2(x))))
    x = self.drop2(x)
    x = self.flatten(x)
    h = torch.relu(self.fc_shared(x))
    h = self.drop3(h)

    class_logits = self.fc_class(h)
    phys_preds = self.fc_phys(h)

    return class_logits, phys_preds

class PhysicalFeatureHeadFCN(nn.Module):
  def __init__(self,in_channels = 6, conv1_out = 64, conv2_out = 128, conv3_out = 128, fc_hidden = 128, dropout = 0.3, num_classes = 12):
    super(PhysicalFeatureHeadFCN, self).__init__()
    # ——— convolutional backbone ———
    self.conv1 = nn.Conv1d(in_channels, conv1_out, kernel_size=8, padding=4)
    self.bn1   = nn.BatchNorm1d(conv1_out)

    self.conv2 = nn.Conv1d(conv1_out, conv2_out, kernel_size=5, padding=2)
    self.bn2   = nn.BatchNorm1d(conv2_out)

    self.conv3 = nn.Conv1d(conv2_out, conv3_out, kernel_size=3, padding=1)
    self.bn3   = nn.BatchNorm1d(conv3_out)

    self.global_pool = nn.AdaptiveAvgPool1d(1)

    # ——— shared fully-connected layer ———
    self.fc_shared = nn.Linear(conv3_out, fc_hidden)
    self.drop      = nn.Dropout(dropout)

    # ——— output heads ———
    self.fc_class = nn.Linear(fc_hidden, num_classes)  # classification head
    self.fc_phys  = nn.Linear(fc_hidden, 3)            # regression head for [mass, velocity, decel]

  def forward(self, x):
    # x: (batch, time, features) → (batch, features, time)
    x = x.permute(0, 2, 1)

    # conv stack
    x = torch.relu(self.bn1(self.conv1(x)))
    x = torch.relu(self.bn2(self.conv2(x)))
    x = torch.relu(self.bn3(self.conv3(x)))

    # global pooling → (batch, conv3_out, 1) → (batch, conv3_out)
    x = self.global_pool(x).squeeze(-1)

    # shared dense + dropout
    h = torch.relu(self.fc_shared(x))
    h = self.drop(h)

    # two heads
    class_logits = self.fc_class(h)
    phys_preds   = self.fc_phys(h)
    return class_logits, phys_preds


## **1.2 Preparing dataset**

In this section the focus is preparing the data set for training. This includes data augmentation functions and feature abaliation.

In [5]:
### Funtions for augmentation ###
def oversample_data(df, oversample_fn, oversample_times, seed=None):
  """
  Select rows where oversample_fn(row) is True, then:
    - duplicate them floor(oversample_times) times
    - plus sample the fractional remainder (if any)
  """
  # which rows to oversample
  df_sel = df[df.apply(oversample_fn, axis=1)]
  if oversample_times <= 0 or df_sel.empty:
    return np.empty((0,)), np.empty((0,))

  # split into integer & fractional parts
  n_full = int(oversample_times)
  frac = oversample_times - n_full

  parts = [df]
  if n_full > 0:
    parts += [df_sel] * n_full
  if frac > 0:
    parts.append(df_sel.sample(frac=frac, random_state=seed).reset_index(drop=True))

  df_dup = pd.concat(parts, ignore_index=True)
  return df_dup

def add_noise(data, noise_level=0.001):
  # generate Gaussian noise and add to input data
  noise = noise_level * np.random.randn(*data.shape)
  return data + noise

def time_shift(data, shift_max=5):
  # choose a random integer shift in range [-shift_max, shift_max)
  shift = np.random.randint(-shift_max, shift_max)
  if shift > 0:
    return np.pad(data, ((0,0), (shift,0), (0,0)), mode='constant')[:, :-shift, :]
  elif shift < 0:
    return np.pad(data, ((0,0), (0,-shift), (0,0)), mode='constant')[:, -shift:, :]
  else:
    return data

def scale_data(data, scale_range=(0.5, 2), accelerations_to_scale = range(6)):
  # sample scale factors for each sample in batch
  factor_vector = np.random.uniform(*scale_range, size = data.shape[0])
  for i in accelerations_to_scale:
    data[:, :, i] *= factor_vector[:, np.newaxis]
  return data, factor_vector

def truncate_beginning_signals(df, portion, signal_col = "sensor_data"):
  if not (0 <= portion < 1):
    raise ValueError("`pct` must be in [0,1).")
  def _truncate(arr):
    ts = arr.shape[0]
    cutoff = int(ts * portion)
    return arr[cutoff:, :]
  new_df = df.copy()
  new_df[signal_col] = new_df[signal_col].apply(_truncate)
  return new_df


def sliding_windows(df, seq_col, label_col, window_length, stride, phys_cols=('mass','velocity','deceleration_average')):
  """
  Slice each time-series into overlapping windows and collect labels and physical features.

  Returns:
  -------
  windows : np.ndarray, shape (N_windows, window_length, n_features)
  labels  : np.ndarray, shape (N_windows,)
  phys    : np.ndarray, shape (N_windows, len(phys_cols))
  parents : np.ndarray, shape (N_windows,)
  """

  X_list, y_list, phys_list, parent_list = [], [], [], []
  X_list, y_list, phys_list, parent_list = [], [], [], []

  for parent_id, row in df.iterrows():
    seq = row[seq_col]                  # shape (T, n_features)
    T = seq.shape[0]

    # collect all full-length windows
    windows = [
      seq[start:start+window_length]
      for start in range(0, T - window_length + 1, stride)
    ]
    if not windows:
      continue

    X_win = np.stack(windows, axis=0)   # (n_wins, window_length, n_features)
    n_wins = X_win.shape[0]

    # append data
    X_list.append(X_win)
    y_list.append(np.full(n_wins, row[label_col]))

    vals = [row[c] for c in phys_cols]  # e.g. [mass, velocity, decel_avg]
    phys_list.append(np.repeat([vals], n_wins, axis=0))

    parent_list.append(np.full(n_wins, parent_id))

  # concatenate everything into flat arrays
  windows = np.concatenate(X_list, axis=0)
  labels  = np.concatenate(y_list, axis=0)
  phys    = np.concatenate(phys_list, axis=0)
  parents = np.concatenate(parent_list, axis=0)

  return windows, labels, phys, parents


In the following function we prepare the dataset for running the experiments.

In [6]:
def prepare_dataset(
  df,train_fraction=0.8, return_split=True, seed=None,
  truncation = False, truncation_portion = 0.3,
  oversample = False, oversample_fn = None, oversample_times = 1,
  sliding = False, window_length = 32, stride = 16,
  standarize = False,
  normalization = False,
  noise = False, noise_level = 0.001,
  shift = False, shift_max = 5,
  scale = False, scale_range = (0.5, 2), accelerations_to_scale = range(6), phys_scale_factors = None,
  warp = False, warp_n_speed_change = 2, warp_max_speed_ratio = 2,
  augment_filter_fn = None,
  feature_ablation_flag = False,
  drop_channels = [],
  select_model = "LSTMClassifier"
  ):

  assert select_model in model_list, f"Invalid model name. Choose from {model_list}"
  assert not (feature_ablation_flag and drop_channels == range(5)), f"Cannot remove all channels"
  assert not (standarize and normalization), "Standardization and normalization cannot be used together"

  #Remove rows with mass, velocity, or deceleration average equal to 0 (it breaks the MSE later on if they are equal to 0)
  if select_model in ["PhysicalFeatureHeadLSTM", "PhysicalFeatureHeadCNN", "PhysicalFeatureHeadFCN"]:
    df = df[(df["mass"] != 0) & (df["velocity"] != 0) & (df["deceleration_average"] != 0)]

  phys_cols = ["mass", "velocity", "deceleration_average"]

  if truncation:
    df = truncate_beginning_signals(df, truncation_portion)

  if oversample:
    df = oversample_data(df, oversample_fn, oversample_times)

  if sliding:
    train_data, train_labels, train_phys, _ = sliding_windows(df, "sensor_data", "label", window_length, stride)
  else:
    train_data = np.stack(df.sensor_data.to_numpy())
    train_labels = df.label.to_numpy()
    train_phys = df[phys_cols].to_numpy()

  #Scalers
  if select_model in ["PhysicalFeatureHeadLSTM", "PhysicalFeatureHeadCNN", "PhysicalFeatureHeadFCN"]:
    phys_scaler = StandardScaler().fit(train_phys)
    phys_scaled = phys_scaler.transform(train_phys)
  else:
    phys_scaler = None

  scaler = None
  feature_max = None
  feature_min = None

  X_raw = train_data.copy()
  X_train = train_data.copy()
  Y_train = train_labels.copy()
  if select_model in ["PhysicalFeatureHeadLSTM", "PhysicalFeatureHeadCNN", "PhysicalFeatureHeadFCN"]:
    phys_all = phys_scaled.copy()

  #Just augmentate the rows that pass the augmentation_fn filter
  if augment_filter_fn is not None:
    df_to_augment = df[df.apply(augment_filter_fn, axis = 1)]
  else:
    df_to_augment = df

  if df_to_augment.empty:
    raise ValueError("No data to augment")

  if sliding:
    X_aug_raw, Y_aug_raw, phys_aug, _ = sliding_windows(df_to_augment, "sensor_data", "label", window_length, stride)
  else:
    X_aug_raw = np.stack(df_to_augment.sensor_data.to_numpy())
    Y_aug_raw = df_to_augment.label.to_numpy()
    phys_aug = df_to_augment[phys_cols].to_numpy()

  if select_model in ["PhysicalFeatureHeadLSTM", "PhysicalFeatureHeadCNN", "PhysicalFeatureHeadFCN"]:
    phys_aug = phys_scaler.transform(phys_aug)

  if noise:
    X_noise = add_noise(X_aug_raw, noise_level = noise_level)
    X_train = np.concatenate([X_train, X_noise])
    Y_train = np.concatenate([Y_train, Y_aug_raw])
    if select_model in ["PhysicalFeatureHeadLSTM", "PhysicalFeatureHeadCNN", "PhysicalFeatureHeadFCN"]:
      phys_all = np.concatenate([phys_all, phys_aug])

  if warp:
    augmenter = TimeWarp(n_speed_change=warp_n_speed_change, max_speed_ratio=warp_max_speed_ratio)
    X_warp = augmenter.augment(X_aug_raw)
    X_train = np.concatenate([X_train, X_warp])
    Y_train = np.concatenate([Y_train, Y_aug_raw])
    if select_model in ["PhysicalFeatureHeadLSTM", "PhysicalFeatureHeadCNN", "PhysicalFeatureHeadFCN"]:
      phys_all = np.concatenate([phys_all, phys_aug])

  if shift:
    X_shift = time_shift(X_aug_raw, shift_max = shift_max)
    X_train = np.concatenate([X_train, X_shift])
    Y_train = np.concatenate([Y_train, Y_aug_raw])
    if select_model in ["PhysicalFeatureHeadLSTM", "PhysicalFeatureHeadCNN", "PhysicalFeatureHeadFCN"]:
      phys_all = np.concatenate([phys_all, phys_aug])

  if scale:
    X_scale, factor_vector = scale_data(X_aug_raw, scale_range = scale_range, accelerations_to_scale = accelerations_to_scale)
    X_train = np.concatenate([X_train, X_scale])
    Y_train = np.concatenate([Y_train, Y_aug_raw])
    if select_model in ["PhysicalFeatureHeadLSTM", "PhysicalFeatureHeadCNN", "PhysicalFeatureHeadFCN"]:
      psf = np.array(phys_scale_factors)
      phys_scale = phys_aug * (factor_vector[:, None] * psf)
      phys_all = np.concatenate([phys_all, phys_scale])

  # Remove selected sensor(s)
  if feature_ablation_flag:
    X_train = np.delete(X_train, drop_channels, axis=2)

  if standarize:
    #Get shapes
    n_train, time_steps, channels = X_train.shape
    #Reshape
    train_reshaped = X_train.reshape(-1, channels)
    scaler = StandardScaler()
    scaler.fit(train_reshaped)
    data_scaled = scaler.transform(train_reshaped).reshape(n_train, time_steps, channels)
  else:
    data_scaled = X_train

  if normalization:
    flattened = X_train.reshape(-1, X_train.shape[2])
    feature_min = np.min(flattened, axis=0)
    feature_max = np.max(flattened, axis=0)

    denom = feature_max - feature_min
    denom[denom == 0] = 1e-8

    data_scaled = (X_train - feature_min) / denom
  else:
    data_scaled = X_train

  if select_model in ["LSTMClassifier", "BinaryHeadLSTM", "PhysicalFeatureHeadLSTM", "FCNClassifier", "BinaryHeadFCN", "PhysicalFeatureHeadFCN"]:
    data_scaled = data_scaled
  elif select_model in ["CNNClassifier", "BinaryHeadCNN", "PhysicalFeatureHeadCNN"]:
    data_scaled = np.expand_dims(data_scaled, axis=1)
  else:
    raise ValueError(f"Invalid model name. Choose form {model_list}")

  train_data = torch.tensor(data_scaled, dtype=torch.float32)
  train_labels = torch.tensor(Y_train, dtype=torch.long)
  if select_model in ["PhysicalFeatureHeadLSTM", "PhysicalFeatureHeadCNN", "PhysicalFeatureHeadFCN"]:
    train_phys = torch.tensor(phys_all, dtype=torch.float32)
    dataset = TensorDataset(train_data, train_labels, train_phys)
  else:
    dataset = TensorDataset(train_data, train_labels)

  if return_split:
    if seed is not None:
      torch.manual_seed(seed)
    train_size = int(train_fraction * len(dataset))
    val_size = len(dataset) - train_size
    return random_split(dataset, [train_size, val_size]), scaler, phys_scaler, feature_max, feature_min
  else:
    return dataset, scaler, phys_scaler, feature_max, feature_min

## **1.3 Running experiment**

In this section one can find all the functions regarding the creation of the experiments.

### **1.3.1 Training**

In [7]:
def create_experiment(pr_name = "LSTM", exp_name = "Experiment0"):

  # initiate the comet experiment for tracking
  experiment = comet_ml.Experiment(
                  api_key=COMET_API_KEY,
                  project_name=pr_name
                  )
  experiment.set_name(exp_name)
  # log our hyperparameters, defined above, to the experiment
  for param, value in params.items():
    experiment.log_parameter(param, value)
  experiment.flush()

  return experiment

def run_experiement(params, select_model, train_dataset, val_dataset, experiment_name = "Experiment0", phys_scaler = None):

  # Training and validation split
  train_loader = DataLoader(train_dataset, batch_size=params["batch_size"], shuffle=True)
  val_loader = DataLoader(val_dataset, batch_size=params["batch_size"])

  #Create experiment
  current_experiment = create_experiment(pr_name, experiment_name)

  best_val_loss = float('inf')
  epochs_without_improvement = 0
  train_losses = []
  val_losses = []

  if select_model in ["PhysicalFeatureHeadLSTM", "PhysicalFeatureHeadCNN", "PhysicalFeatureHeadFCN"]:
    train_losses_phys = []
    val_losses_phys = []

  val_accuracies = []

  # Dimension depends on the number of features removed
  InputDim = train_dataset[0][0].shape[1]
  num_classes = 12

  match select_model:
    case "LSTMClassifier":
      model = LSTMClassifier(InputDim, params['hidden_size'], params['num_layers']).to("cuda" if torch.cuda.is_available() else "cpu")
    case "CNNClassifier":
      sample_input = train_dataset[0][0].unsqueeze(0)
      model = CNNClassifier(sample_input.shape, params['conv1_out'], params['dropout'], params['conv2_out'], params['fc_hidden']).to("cuda" if torch.cuda.is_available() else "cpu")
    case "FCNClassifier":
      model = FCNClassifier(InputDim, conv1_out = params['conv1_out'], conv2_out = params['conv2_out'], conv3_out = params['conv3_out']).to("cuda" if torch.cuda.is_available() else "cpu")
    case "BinaryHeadLSTM":
      model = BinaryHeadLSTM(InputDim, params['hidden_size'], params['num_layers']).to("cuda" if torch.cuda.is_available() else "cpu")
    case "BinaryHeadCNN":
      sample_input = train_dataset[0][0].unsqueeze(0)
      model = BinaryHeadCNN(sample_input.shape, params['conv1_out'], params['dropout'], params['conv2_out'], params['fc_hidden']).to("cuda" if torch.cuda.is_available() else "cpu")
    case "BinaryHeadFCN":
      model = BinaryHeadFCN(InputDim, params['conv1_out'], params['conv2_out'], params['conv3_out'], params['fc_hidden'], params['dropout']).to("cuda" if torch.cuda.is_available() else "cpu")
    case "PhysicalFeatureHeadLSTM":
      model = PhysicalFeatureHeadLSTM(InputDim, params['hidden_size'], params['num_layers']).to("cuda" if torch.cuda.is_available() else "cpu")
    case "PhysicalFeatureHeadCNN":
      sample_input = train_dataset[0][0].unsqueeze(0)
      model = PhysicalFeatureHeadCNN(sample_input.shape, params['conv1_out'], params['dropout'], params['conv2_out'], params['fc_hidden']).to("cuda" if torch.cuda.is_available() else "cpu")
    case "PhysicalFeatureHeadFCN":
      model = PhysicalFeatureHeadFCN(InputDim, params['conv1_out'], params['conv2_out'], params['conv3_out'], params['fc_hidden'], params['dropout']).to("cuda" if torch.cuda.is_available() else "cpu")
    case _:
      raise ValueError(f"Invalid model name. Choose from {model_list}")

  criterion = nn.CrossEntropyLoss()
  if select_model in ["PhysicalFeatureHeadLSTM", "PhysicalFeatureHeadCNN", "PhysicalFeatureHeadFCN"]:
    criterion_mse = nn.MSELoss()

  optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay'])

  device = next(model.parameters()).device

  for epoch in range(params["NumberOfEpochs"]):
    model.train()

    if select_model in ["LSTMClassifier", "CNNClassifier", "FCNClassifier"]:
      running_loss = 0.0
      for i, (x_batch, y_batch) in enumerate(train_loader):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

      avg_train_loss = running_loss / len(train_loader)
      train_losses.append(avg_train_loss)
      current_experiment.log_metric("avg_train_loss", avg_train_loss, step=epoch)

      all_preds = []
      all_labels = []

      # Validation step
      model.eval()
      val_loss = 0.0
      correct = 0
      total = 0
      with torch.no_grad():
        for X_batch, y_batch in val_loader:
          X_batch, y_batch = X_batch.to(device), y_batch.to(device)
          outputs = model(X_batch)
          loss = criterion(outputs, y_batch)
          val_loss += loss.item()

          preds = torch.argmax(outputs, dim=1)
          correct += (preds == y_batch).sum().item()
          total += y_batch.size(0)

          all_preds.extend(preds.cpu().numpy())
          all_labels.extend(y_batch.cpu().numpy())

      avg_val_loss = val_loss / len(val_loader)
      val_losses.append(avg_val_loss)
      val_acc = correct / total
      val_accuracies.append(val_acc)
      current_experiment.log_metric("avg_val_loss", avg_val_loss, step=epoch)
      current_experiment.log_metric("val_accuracy", val_acc, step=epoch)

      if params["print_epoch_stats"]:
        print(f"Epoch {epoch+1}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}")

      if params['use_early_stopping']:
        if avg_val_loss < best_val_loss - params['min_delta']:
          best_val_loss = avg_val_loss
          epochs_without_improvement = 0
        else:
          epochs_without_improvement += 1

        if epochs_without_improvement >= params['patience']:
          print(f"Early stopping at epoch {epoch+1}")
          break
    #Physical Features
    elif select_model in ["PhysicalFeatureHeadLSTM", "PhysicalFeatureHeadCNN", "PhysicalFeatureHeadFCN"]:
      running_loss_clf = 0.0
      running_loss_phys = 0.0

      for i, (x_batch, y_batch, phys_batch) in enumerate(train_loader):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        phys_batch = phys_batch.to(device)

        optimizer.zero_grad()
        logits, phys_preds = model(x_batch)
        loss_clf = criterion(logits, y_batch)
        loss_phys = criterion_mse(phys_preds, phys_batch)
        total_loss = loss_clf + params["lambda_phys"] * loss_phys #combined loss

        total_loss.backward()
        optimizer.step()

        running_loss_clf += loss_clf.item()
        running_loss_phys += loss_phys.item()

      avg_train_loss_clf = running_loss_clf / len(train_loader)
      train_losses.append(avg_train_loss_clf)
      avg_train_loss_phys = running_loss_phys / len(train_loader)
      train_losses_phys.append(avg_train_loss_phys)

      current_experiment.log_metric("avg_train_loss_clf", avg_train_loss_clf, step=epoch)
      current_experiment.log_metric("avg_train_loss_phys", avg_train_loss_phys, step=epoch)

      all_preds = []
      all_labels = []

      # Validation step
      model.eval()

      val_loss_clf = 0.0
      val_loss_phys = 0.0

      correct = 0
      total = 0

      all_true_phys = []
      all_pred_phys = []

      with torch.no_grad():
        for X_batch, y_batch, phys_batch in val_loader:
          X_batch, y_batch = X_batch.to(device), y_batch.to(device)
          phys_batch = phys_batch.to(device)

          logits, phys_preds = model(X_batch)
          loss_clf = criterion(logits, y_batch)
          loss_phys = criterion_mse(phys_preds, phys_batch)

          val_loss_clf += loss_clf.item()
          val_loss_phys += loss_phys.item()

          preds = torch.argmax(logits, dim=1)
          correct += (preds == y_batch).sum().item()
          total += y_batch.size(0)

          all_preds.extend(preds.cpu().numpy())
          all_labels.extend(y_batch.cpu().numpy())

          all_true_phys.append(phys_batch.cpu())
          all_pred_phys.append(phys_preds.cpu())

      all_true_phys = torch.cat(all_true_phys, dim=0).numpy()
      all_pred_phys = torch.cat(all_pred_phys, dim=0).numpy()

      if phys_scaler is not None:
        all_true_phys_unscaled = phys_scaler.inverse_transform(all_true_phys)
        all_pred_phys_unscaled = phys_scaler.inverse_transform(all_pred_phys)
      else:
        # if somehow phys_scaler is None, fallback to scaled‐space MAPE
        all_true_phys_unscaled = all_true_phys.copy()
        all_pred_phys_unscaled = all_pred_phys.copy()


      eps = 1e-6

      mass_mape     = 100.0 * np.mean(np.abs((all_true_phys_unscaled[:, 0] - all_pred_phys_unscaled[:, 0]) / (np.abs(all_true_phys_unscaled[:, 0]) + eps)))
      velocity_mape = 100.0 * np.mean(np.abs((all_true_phys_unscaled[:, 1] - all_pred_phys_unscaled[:, 1]) / (np.abs(all_true_phys_unscaled[:, 1]) + eps)))
      decel_mape    = 100.0 * np.mean(np.abs((all_true_phys_unscaled[:, 2] - all_pred_phys_unscaled[:, 2]) / (np.abs(all_true_phys_unscaled[:, 2]) + eps)))


      avg_val_loss_clf = val_loss_clf / len(val_loader)
      val_losses.append(avg_val_loss_clf)

      avg_val_loss_phys = val_loss_phys / len(val_loader)
      val_losses_phys.append(avg_val_loss_phys)

      val_acc = correct / total
      val_accuracies.append(val_acc)

      current_experiment.log_metric("avg_val_loss_clf", avg_val_loss_clf, step=epoch)
      current_experiment.log_metric("avg_val_loss_phys", avg_val_loss_phys, step=epoch)
      current_experiment.log_metric("val_acc", val_acc, step=epoch)

      if params["print_epoch_stats"]:
        print(f"Epoch {epoch+1} — Val Acc: {val_acc*100:.4f}%, "f"Phys MAPE: Mass={mass_mape:.4f}%, Vel={velocity_mape:.4f}%, Decel={decel_mape:.4f}%")

      if params['use_early_stopping']:
        if avg_val_loss_clf < best_val_loss - params['min_delta']:
          best_val_loss = avg_val_loss_clf
          epochs_without_improvement = 0
        else:
          epochs_without_improvement += 1

        if epochs_without_improvement >= params['patience']:
          print(f"Early stopping at epoch {epoch+1}")
          break

    #Binary classification
    elif select_model in ["BinaryHeadLSTM", "BinaryHeadCNN", "BinaryHeadFCN"]:
      running_loss = 0.0
      for i, (x_batch, y_batch) in enumerate(train_loader):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        logits_b, logits_m = model(x_batch)

        # binary loss
        is_broken = (y_batch > 0).long()
        loss_b    = criterion(logits_b, is_broken)

        # multi-class loss on broken only
        idx = (is_broken == 1).nonzero(as_tuple=True)[0]
        if idx.numel() > 0:
          true_lbls = y_batch[idx]              # in 1…11
          pred_m    = logits_m[idx]
          loss_m    = criterion(pred_m, true_lbls)
        else:
          loss_m = torch.tensor(0.0, device=device)

        total_loss = loss_b + params["lambda_multi_head_loss"] * loss_m

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        running_loss += total_loss.item()

      avg_train_loss = running_loss / len(train_loader)
      train_losses.append(avg_train_loss)
      current_experiment.log_metric("avg_train_loss", avg_train_loss, step=epoch)

      all_preds = []
      all_labels = []

      # Validation step
      model.eval()
      running_val_loss = 0.0
      bin_corr = bin_tot = 0
      multi_corr = multi_tot = 0
      with torch.no_grad():
        for X_batch, y_batch in val_loader:
          X_batch, y_batch = X_batch.to(device), y_batch.to(device)
          l_b, l_m = model(X_batch)

          is_broken_val = (y_batch > 0).long()
          loss_b_val = criterion(l_b, is_broken_val)

          idx_val = (is_broken_val == 1).nonzero(as_tuple=True)[0]
          if idx_val.numel() > 0:
            v_loss_m = criterion(l_m[idx_val], y_batch[idx_val])
          else:
            v_loss_m = torch.tensor(0.0, device=device)

          total_loss_val = loss_b_val + params["lambda_multi_head_loss"] * v_loss_m

          running_val_loss += total_loss_val.item()

          #Binary accuracy
          pred_b = l_b.argmax(dim=1)
          bin_corr += (pred_b == is_broken_val).sum().item()
          bin_tot += y_batch.size(0)


          #Multi-class accuracy
          if idx_val.numel() > 0:
            pred_t = l_m[idx_val].argmax(dim=1)
            multi_corr += (pred_t == y_batch[idx_val]).sum().item()
            multi_tot += idx_val.numel()

          bin_acc = bin_corr / bin_tot
          multi_acc = multi_corr / multi_tot if multi_tot > 0 else 0.0

      avg_val_loss = running_val_loss / len(val_loader)
      val_losses.append(avg_val_loss)
      current_experiment.log_metric("avg_val_loss", avg_val_loss, step=epoch)

      bin_acc   = bin_corr / bin_tot
      val_acc = multi_corr / multi_tot if multi_tot > 0 else 0.0

      if params["print_epoch_stats"]:
        print(f"Epoch {epoch+1}: "f"Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}, "f"Bin Acc={bin_acc*100:.2f}%, Multi Acc={val_acc*100:.2f}%")

      if params['use_early_stopping']:
        if avg_val_loss < best_val_loss - params['min_delta']:
          best_val_loss = avg_val_loss
          epochs_without_improvement = 0
        else:
          epochs_without_improvement += 1

        if epochs_without_improvement >= params['patience']:
          print(f"Early stopping at epoch {epoch+1}")
          break

      current_experiment.log_metric("val_acc", val_acc, step=epoch)

  return model, device, current_experiment, val_acc

### **1.3.2 Testing**

#### **1.3.2.1 Prediction Analysis**

In [8]:
#Functions to analyze results

def analyze_predictions(dfs):
  # Concatenate all DataFrames into one
  df = pd.concat(dfs, ignore_index=True)

  # Ensure correct types
  df['label'] = df['label'].astype(int)
  df['predicted_label'] = df['predicted_label'].astype(int)

  sns.set(style="whitegrid")
  fig = plt.figure(figsize=(20, 40))

  # 1. Confusion Matrix (Multiclass)
  plt.subplot(5, 1, 1)
  cm = confusion_matrix(df['label'], df['predicted_label'], labels=range(12))
  ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
              xticklabels=range(12),
              yticklabels=range(12),
              cbar = True)
  # Highlight diagonal elements
  for i in range(cm.shape[0]):
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False, edgecolor='green', lw=2))

  plt.title('Confusion Matrix (True: label, Predicted: predicted_label)')
  plt.xlabel('Predicted Label')
  plt.ylabel('True Label')

  # 2. Accuracy per Model Variant
  plt.subplot(5, 1, 2)
  acc_by_model = df.groupby('model', observed=False).apply(
      lambda x: (x['label'] == x['predicted_label']).mean()
  ).reset_index(name='accuracy')
  acc_by_model = acc_by_model.sort_values(by='accuracy', ascending=False)
  sns.barplot(data=acc_by_model, x='model', y='accuracy', palette='Blues_d')
  plt.title('Accuracy by Model Variant')
  plt.xticks(rotation=90)
  plt.ylabel('Accuracy')
  plt.xlabel('Model Variant')

  # 3. Accuracy by Unique Mass Values
  plt.subplot(5, 1, 3)
  acc_by_mass = df.groupby('mass', observed=True).apply(
      lambda x: (x['label'] == x['predicted_label']).mean()
  ).reset_index(name='accuracy')
  acc_by_mass = acc_by_mass.sort_values(by='mass')
  sns.barplot(data=acc_by_mass, x='mass', y='accuracy', color='salmon')
  plt.title('Accuracy by Mass')
  plt.xticks(rotation=90)
  plt.xlabel('Mass')
  plt.ylabel('Accuracy')

  # 4. Accuracy by Unique Velocity Values
  plt.subplot(5, 1, 4)
  acc_by_velocity = df.groupby('velocity', observed=True).apply(
      lambda x: (x['label'] == x['predicted_label']).mean()
  ).reset_index(name='accuracy')
  acc_by_velocity = acc_by_velocity.sort_values(by='velocity')
  sns.barplot(data=acc_by_velocity, x='velocity', y='accuracy', color='seagreen')
  plt.title('Accuracy by Velocity')
  plt.xlabel('Velocity')
  plt.ylabel('Accuracy')

  # 5. Accuracy by Binned Deceleration Average (10 bins)
  plt.subplot(5, 1, 5)
  df['decel_bin'] = pd.cut(df['deceleration_average'], bins=10)
  acc_by_decel_bin = df.groupby('decel_bin').apply(
      lambda x: (x['label'] == x['predicted_label']).mean()
  ).reset_index(name='accuracy')
  acc_by_decel_bin['decel_bin_str'] = acc_by_decel_bin['decel_bin'].astype(str)
  sns.barplot(data=acc_by_decel_bin, x='decel_bin_str', y='accuracy', color='purple')
  plt.xticks(rotation=45, ha='right')
  plt.title('Accuracy by Deceleration Average (Binned)')
  plt.xlabel('Deceleration Average Bin')
  plt.ylabel('Accuracy')

  plt.tight_layout()

  return fig


#### **1.3.2.2 Testing model**

In [9]:
def testing_model(df_test, params, select_model, model, device, current_experiment, scaler = None, phys_scaler = None, feature_min = None, feature_max = None):

  # Prepare data tensors
  if params["truncation"]:
    df_test = truncate_beginning_signals(df_test, params["truncation_portion"])

  if params["sliding"]:
    test_data, test_labels, _, parents = sliding_windows(df_test, "sensor_data", "label", params["window_length"], params["stride"])
    original_test_labels = df_test.label.to_numpy()
  else:
    test_data = np.stack(df_test.sensor_data.to_numpy())
    test_labels = df_test.label.to_numpy()

  if params["feature_ablation_flag"]:
    test_data = np.delete(test_data, params["sensor_idx"], axis=2)

  if scaler is not None:
    n_test, time_steps, channels = test_data.shape
    test_reshaped = test_data.reshape(-1, channels)
    test_scaled = scaler.transform(test_reshaped).reshape(n_test, time_steps, channels)
  else:
    test_scaled = test_data

  if params["normalization"]:
    test_scaled = (test_data - feature_min)/(feature_max - feature_min)
  else:
    test_scaled = test_data


  if select_model in ["LSTMClassifier", "BinaryHeadLSTM", "PhysicalFeatureHeadLSTM", "FCNClassifier", "BinaryHeadFCN", "PhysicalFeatureHeadFCN"]:
    test_scaled = test_scaled
  elif select_model in ["CNNClassifier", "BinaryHeadCNN", "PhysicalFeatureHeadCNN"]:
    test_scaled = np.expand_dims(test_scaled, axis=1)
  elif select_model not in model_list:
    raise ValueError(f"Invalid model name. Choose form {model_list}")

  test_data = torch.tensor(test_scaled, dtype=torch.float32)
  test_labels = torch.tensor(test_labels, dtype=torch.long)

  # Wrap in DataLoader
  test_dataset = TensorDataset(test_data, test_labels)
  test_loader = DataLoader(test_dataset, batch_size=params["batch_size"])

  # Evaluate the model
  model.eval()
  all_test_preds = []
  all_test_labels = []

  if select_model in ["PhysicalFeatureHeadLSTM", "PhysicalFeatureHeadCNN", "PhysicalFeatureHeadFCN"]:
    all_test_masses = []
    all_test_velocities = []
    all_test_decelerations = []

  with torch.no_grad():
    for X_batch, y_batch in test_loader:
      X_batch = X_batch.to(device)
      y_batch = y_batch.to(device)

      if select_model in ["PhysicalFeatureHeadLSTM", "PhysicalFeatureHeadCNN", "PhysicalFeatureHeadFCN"]:
        outputs, phys_preds = model(X_batch)
        preds = torch.argmax(outputs, dim=1)
        phys_np = phys_preds.cpu().numpy()
        all_test_masses      .extend(phys_np[:, 0])
        all_test_velocities  .extend(phys_np[:, 1])
        all_test_decelerations.extend(phys_np[:, 2])

      elif select_model in ["BinaryHeadLSTM", "BinaryHeadCNN", "BinaryHeadFCN"]:
        logits_b, logits_m = model(X_batch)
        pred_broken = logits_b.argmax(dim=1)
        pred_type = logits_m.argmax(dim=1)
        preds = torch.where(pred_broken == 1, pred_type, torch.zeros_like(pred_broken))

      else:
        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1)

      all_test_preds.extend(preds.cpu().numpy())
      all_test_labels.extend(y_batch.cpu().numpy())

  if params["sliding"]:
    # collect preds per parent
    votes = defaultdict(list)
    # also remember the one true label for each parent
    truth = {}

    for pred, true_lbl, p in zip(all_test_preds, all_test_labels, parents):
      votes[p].append(pred)
      truth[p] = true_lbl   # they are all the same within one parent

    # now build new lists, in sorted‐parent order (or any consistent order)
    parent_ids = sorted(votes.keys())
    maj_preds  = [max(set(votes[p]), key=votes[p].count) for p in parent_ids]
    maj_truths = [truth[p] for p in parent_ids]

    # overwrite for metric calculation
    all_test_preds  = maj_preds
    all_test_labels = maj_truths

    if select_model in ["PhysicalFeatureHeadLSTM", "PhysicalFeatureHeadCNN", "PhysicalFeatureHeadFCN"]:
      # average phys per parent
      def agg(list_vals):
        d = {}
        for v, p in zip(list_vals, parents):
          d.setdefault(p, []).append(v)
        return [float(np.mean(d[p])) for p in parent_ids]

      all_test_masses       = agg(all_test_masses)
      all_test_velocities   = agg(all_test_velocities)
      all_test_decelerations= agg(all_test_decelerations)

  labels_present = sorted(list(set(all_test_labels)))
  # Metrics
  acc = accuracy_score(all_test_labels, all_test_preds)
  macro_f1 = f1_score(all_test_labels, all_test_preds, average='macro', labels = labels_present)
  report = classification_report(all_test_labels, all_test_preds, output_dict=False)

  # Logging
  print(f"\nTest Accuracy: {acc:.4f}")
  print(f"Macro F1 Score: {macro_f1:.4f}")

  current_experiment.log_metric("test_accuracy", acc)
  current_experiment.log_metric("test_macro_f1", macro_f1)
  current_experiment.log_text("test_classification_report", report)

  labels = [str(i) for i in range(12)]
  current_experiment.log_confusion_matrix(
      y_true=all_test_labels,
      y_predicted=all_test_preds,
      labels=labels,
      title="Confusion Matrix on Test Set"
  )


  #log graphs to help understand the results

  df_test_with_preds = df_test.copy()
  df_test_with_preds["predicted_label"] = all_test_preds
  if select_model in ["PhysicalFeatureHeadLSTM", "PhysicalFeatureHeadCNN", "PhysicalFeatureHeadFCN"]:
    df_test_with_preds["predicted_mass"] = all_test_masses
    df_test_with_preds["predicted_velocity"] = all_test_velocities
    df_test_with_preds["predicted_deceleration"] = all_test_decelerations

  plt.ioff()
  fig = analyze_predictions([df_test_with_preds])
  current_experiment.log_figure(figure_name="Test Set Predictions", figure=fig)

  plt.close(fig)

  current_experiment.end()

  return acc, macro_f1, df_test_with_preds


### **Putting all together**

In this section we combine all the functions above to run consecutive experiments to get an avarage understanding of how the model performs.

In [10]:
def run_random_split_experiments(df, df_test, params, select_model, experiment_name):

  iterations = params["iterations"]

  all_val_accuracies = []

  all_test_accuracies = []
  all_test_f1_scores = []

  dfs_test_with_preds = []


  for i in range(iterations):
    print(f"\n--- Random Split Iteration {i+1}/{iterations} ---")

    (train_dataset, val_dataset), scaler, phys_scaler, feature_min, feature_max = prepare_dataset(
      df,
      train_fraction=0.8, return_split=True, seed=None,
      truncation = params["truncation"], truncation_portion = params["truncation_portion"],
      oversample = params["oversample"], oversample_fn = params["oversample_fn"], oversample_times = params["oversample_times"],
      sliding = params["sliding"], window_length = params["window_length"], stride = params["stride"],
      noise = params["noise"], noise_level = params["noise_level"],
      standarize = params["standarize"],
      normalization = params["normalization"],
      scale = params["scale"], scale_range = params["scale_range"], accelerations_to_scale = params["accelerations_to_scale"], phys_scale_factors = params["phys_scale_factors"],
      warp = params["warp"], warp_n_speed_change = params["warp_n_speed_change"], warp_max_speed_ratio = params["warp_max_speed_ratio"],
      shift = params["shift"], shift_max = params["shift_max"],
      augment_filter_fn = params["augment_filter_fn"],
      feature_ablation_flag = params["feature_ablation_flag"],
      drop_channels = params['sensor_idx'],
      select_model = select_model
      )

    model, device, current_experiment, val_acc = run_experiement(
      params = params,
      select_model = select_model,
      train_dataset = train_dataset,
      val_dataset = val_dataset,
      experiment_name = experiment_name,
      phys_scaler = phys_scaler
      )

    all_val_accuracies.append(val_acc)

    test_acc, test_macro_f1, df_test_with_preds_temp = testing_model(
        df_test = df_test,
        params = params,
        select_model = select_model,
        model = model,
        device = device,
        current_experiment = current_experiment,
        scaler = scaler,
        phys_scaler = phys_scaler,
        feature_min = feature_min,
        feature_max = feature_max
        )

    dfs_test_with_preds.append(df_test_with_preds_temp)

    all_test_accuracies.append(test_acc)
    all_test_f1_scores.append(test_macro_f1)

  avg_test_acc = sum(all_test_accuracies) / len(all_test_accuracies)
  avg_test_f1 = sum(all_test_f1_scores) / len(all_test_f1_scores)
  avg_val_acc = sum(all_val_accuracies) / len(all_val_accuracies)

  print(f"Average Test Accuracy over {iterations} random splits: {avg_test_acc:.4f}")
  print(f"Average Test F1 Score over {iterations} random splits: {avg_test_f1:.4f}")
  print(f"Average Validation Accuracy over {iterations} random splits: {avg_val_acc:.4f}")

  return all_test_accuracies, all_test_f1_scores, all_val_accuracies, avg_test_acc, avg_test_f1, avg_val_acc, dfs_test_with_preds

# **Main**

In [ ]:
# Who's running the code?
user = "Nikita"

# Choose a model from the list
model_list = [
    "LSTMClassifier",
    "CNNClassifier",
    "FCNClassifier",
    "BinaryHeadLSTM",
    "BinaryHeadCNN",
    "BinaryHeadFCN",
    "PhysicalFeatureHeadLSTM",
    "PhysicalFeatureHeadCNN",
    "PhysicalFeatureHeadFCN"
    ]

chosen_model = "CNNClassifier"

# Comet specifications
COMET_API_KEY = API_dict[user]
pr_name = "final code trials"
experiment_name = "Exp {}".format(chosen_model)

print(pr_name)
print(experiment_name)

(all_test_accuracies,
 all_test_f1_scores,
 all_val_accuracies,
 avg_test_acc,
 avg_test_f1,
 avg_val_acc,
 dfs_test_with_preds) = run_random_split_experiments(df, df_test, params, chosen_model, experiment_name)


# ensure the target folder exists

#Results direction
res_dir = PROJECT_ROOT + '/results'
os.makedirs(res_dir, exist_ok=True)
outpath = os.path.join(res_dir, "{}_prediction_analysis.png".format(chosen_model))
fig = analyze_predictions(dfs_test_with_preds)
fig.savefig(outpath, bbox_inches="tight")
print("Results saved in {}".format(outpath))

# **VAE**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Prepare the dataset: split into train/val with specific model shape
(train_dataset, val_dataset), scaler, phys_scaler, feature_max, feature_min = prepare_dataset(
    df,
    train_fraction=0.8,
    return_split=True,
    seed=42,
    standarize=False,
    noise=False,
    scale=False,
    warp=False,
    shift=False,
    augment_filter_fn=None,
    feature_ablation_flag=False,
    drop_channels=False,
    select_model="FCNClassifier"  # Defines the shape of X (e.g., FCN input format)
)

# Determine input feature dimension
sample_X, _ = train_dataset[0]
input_dim = sample_X.shape[1]

# Define batch size and DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size)


# Define FCN-based Debiasing Variational Autoencoder (DB-VAE)
class FCN_DBVAEClassifier(nn.Module):
    def __init__(self, in_channels=6, seq_len=128, latent_dim=16, num_classes=12):
        super().__init__()
        # Encoder: 3-layer FCN with global average pooling
        self.encoder_conv = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=8, padding=4),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.flatten = nn.Flatten()
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        # Decoder: reconstruct input from latent space
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, seq_len * in_channels),
        )

        # Classifier: predict class from latent space
        self.classifier = nn.Linear(latent_dim, num_classes)

    def encode(self, x):
        x = x.permute(0, 2, 1)  # Switch to [B, Channels, Time]
        x = self.encoder_conv(x)
        h = self.flatten(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = (0.5 * logvar).exp()
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        x_rec = self.decoder(z)
        x_rec = x_rec.view(-1, 128, 6)
        return x_rec

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_rec = self.decode(z)
        y_logit = self.classifier(z)
        return x_rec, mu, logvar, y_logit


# Evaluate accuracy and macro F1 score on given dataloader
def evaluate_model(model, dataloader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            _, _, _, y_logit = model(x_batch)
            pred = torch.argmax(y_logit, dim=1)
            y_true.extend(y_batch.cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    present_labels = sorted(set(y_true))  # Use only labels that appear
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro", labels=present_labels)
    return acc, f1

# Get predictions for visualization
def get_predictions(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            _, _, _, y_logit = model(x_batch)
            pred = torch.argmax(y_logit, dim=1)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    return all_labels, all_preds


# Train the model once using a specific seed
def train_one_run(seed, train_dataset, val_dataset, test_dataset, input_dim):
    torch.manual_seed(seed)

    # DataLoaders for each split
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=64)
    test_loader  = DataLoader(test_dataset, batch_size=64)

    # Initialize model
    model = FCN_DBVAEClassifier(in_channels=6, seq_len=128, latent_dim=16, num_classes=12).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)
    beta = 1e-3
    lambda_clf = 1.0
    epochs = 100
    early_stop_patience = 10

    # Early stopping trackers
    best_val_acc = 0
    best_val_f1 = 0
    best_model_state = None
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        running_total = 0.0
        running_rec   = 0.0
        running_kl    = 0.0
        running_clf   = 0.0

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            # Forward pass
            x_rec, mu, logvar, y_logit = model(x_batch)

            # Losses
            rec_loss = nnf.mse_loss(x_rec, x_batch, reduction='mean')
            kl_loss  = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            loss_vae = rec_loss + beta * kl_loss
            clf_loss = nnf.cross_entropy(y_logit, y_batch)

            # Total loss
            loss = loss_vae + lambda_clf * clf_loss

            # Backpropagation
            opt.zero_grad()
            loss.backward()
            opt.step()

            # Logging
            running_rec   += rec_loss.item()
            running_kl    += kl_loss.item()
            running_clf   += clf_loss.item()
            running_total += loss.item()

        # Averaged training loss
        avg_rec = running_rec / len(train_loader)
        avg_kl  = running_kl / len(train_loader)
        avg_clf = running_clf / len(train_loader)

        # Validation accuracy and macro F1
        val_acc, val_f1 = evaluate_model(model, val_loader)
        print(f"Epoch {epoch+1}/{epochs} — Loss: {running_total/len(train_loader):.4f} — Rec: {avg_rec:.4f} — KL: {avg_kl:.4f} — Clf: {avg_clf:.4f} — Val Acc: {val_acc:.4f} — Macro F1: {val_f1:.4f}")

        # Early stopping check
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_f1 = val_f1
            best_model_state = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= early_stop_patience:
                print("Early stopping triggered.")
                break

    # Load best model and evaluate on test data
    model.load_state_dict(best_model_state)
    test_acc, test_f1 = evaluate_model(model, test_loader)
    test_labels, test_preds = get_predictions(model, test_loader)

    # Save predictions for visualization
    test_df = df_test.copy().reset_index(drop=True)
    test_df["label"] = test_labels
    test_df["predicted_label"] = test_preds
    test_df["model"] = f"run_{seed}"

    print(f"Final Test Accuracy: {test_acc:.4f} — Macro F1: {test_f1:.4f}\n")
    return test_acc, test_f1, test_df


# Prepare test dataset
test_dataset, *_ = prepare_dataset(
    df_test,
    return_split=False,
    standarize=False,
    noise=False,
    scale=False,
    warp=False,
    shift=False,
    augment_filter_fn=None,
    feature_ablation_flag=False,
    drop_channels=False,
    select_model="FCNClassifier"
)

# Run the training process 5 times with different seeds
test_accuracies = []
test_f1s = []
dfs = []

for run in range(10):
    seed = 42 + run
    print(f"Run {run+1} with seed {seed}")
    acc, f1, df_preds = train_one_run(seed, train_dataset, val_dataset, test_dataset, input_dim)
    test_accuracies.append(acc)
    test_f1s.append(f1)
    dfs.append(df_preds)

# Print summary statistics over 10 runs
print("\n10-run average results:")
print(f"   Avg Test Accuracy: {np.mean(test_accuracies):.4f} ± {np.std(test_accuracies):.4f}")
print(f"   Avg Macro F1     : {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")



res_dir = PROJECT_ROOT + '/resultsVAE'
os.makedirs(res_dir, exist_ok=True)
outpath = os.path.join(res_dir, "vae_prediction_analysis.png")
fig = analyze_predictions(dfs)
fig.savefig(outpath, bbox_inches="tight")
print("Results saved in {}".format(outpath))
